In [85]:
import numpy as np
import igl
import meshplot as mp
from scipy.spatial.transform import Rotation
import ipywidgets as iw
import time
import scipy.sparse as sp


In [86]:
WOODY = "data/woody-hi.off"
WOODY_LABEL = 'data/woody-hi.label.npy'
HAND = 'data/hand.off'
HAND_LABEL = 'data/hand.label.npy'


v, f = igl.read_triangle_mesh(HAND)
labels = np.load(HAND_LABEL).astype(int)
v -= v.min(axis=0)
v /= v.max()

In [87]:
def remove_hfdetails(v, f, labels):
    # set up matrices
    Lw = -igl.cotmatrix(v, f)  # laplacian
    M = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_BARYCENTRIC)   # mass matrix 
    M_inv = sp.diags(1 / M.diagonal())
    
    # use labels
    free = np.where(labels == 0)[0]
    handle = np.where(labels > 0)[0]
    
    A = Lw @ M_inv @ Lw 
    Aff = A[free, :][:, free]
    Afc = A[free, :][:, handle]
    
    handle_vert_pos = v[labels>0, :]

    rhs = -Afc @ handle_vert_pos
    x = sp.linalg.spsolve(Aff, rhs)
    
    v_smooth = v.copy()
    v_smooth[free] = x
    
    return v_smooth


In [88]:
def deform_smooth_mesh(base_smooth, f, labels, handle_vertex_positions):
    Lw = -igl.cotmatrix(base_smooth, f)
    M = igl.massmatrix(base_smooth, f, igl.MASSMATRIX_TYPE_BARYCENTRIC)
    M_inv = sp.diags(1 / M.diagonal())
    
    A = Lw @ M_inv @ Lw
    free = np.where(labels == 0)[0]
    handle = np.where(labels > 0)[0]

    Aff = A[free, :][:, free]
    Afc = A[free, :][:, handle]
    
    vert_pos = handle_vertex_positions[handle, :]

    rhs = -Afc @ vert_pos
    X = np.zeros((len(free), 3))
    for i in range(3):
        X[:, i] = sp.linalg.spsolve(Aff, rhs[:, i])

    deformed = base_smooth.copy()
    deformed[free, :] = X
    deformed[handle, :] = vert_pos
    return deformed


In [89]:
def compute_detail_vectors(orig_vertices, verttices, faces):
    smooth_vertices = verttices.copy()
    d = orig_vertices - smooth_vertices
    normals = igl.per_vertex_normals(smooth_vertices, faces)
    adjacency = igl.adjacency_list(faces)
    edge_indices = np.zeros(orig_vertices.shape[0], dtype=np.int32)
    detail_vecs = np.zeros_like(orig_vertices)

    for vidx in range(orig_vertices.shape[0]):
        normal = normals[vidx]
        neighbor_verts = smooth_vertices[adjacency[vidx]] - smooth_vertices[vidx]
        # rremove normal 
        proj = neighbor_verts - (neighbor_verts @ normal)[:, None] * normal
        proj_lengths = np.linalg.norm(proj, axis=1)

        if np.all(proj_lengths < 1e-8):
            tangent = np.array([1.0, 0.0, 0.0])
            bitangent = np.cross(normal, tangent)
            edge_indices[vidx] = adjacency[vidx][0] if len(adjacency[vidx]) > 0 else vidx
        else:
            max_idx = np.argmax(proj_lengths)
            edge_indices[vidx] = adjacency[vidx][max_idx]
            tangent = proj[max_idx] / np.linalg.norm(proj[max_idx])
            bitangent = np.cross(normal, tangent)
        
        # coefficients
        detail_vecs[vidx] = [
            np.dot(d[vidx], tangent),
            np.dot(d[vidx], bitangent),
            np.dot(d[vidx], normal)
        ]
    return detail_vecs, edge_indices


In [90]:
def apply_detail_to_deformed(deformed_base, faces, encoded_details, edge_neighbors):
    normals_new = igl.per_vertex_normals(deformed_base, faces)
    mesh_with_detail = deformed_base.copy()

    edge_vectors = deformed_base[edge_neighbors] - deformed_base
    edge_proj = edge_vectors - np.sum(edge_vectors * normals_new, axis=1, keepdims=True) * normals_new
    edge_lengths = np.linalg.norm(edge_proj, axis=1)
    

    tangent_dirs = np.zeros_like(normals_new)
    for idx in range(normals_new.shape[0]):
        if edge_lengths[idx] > 1e-8:
            tangent_dirs[idx] = edge_proj[idx] / edge_lengths[idx]
        else:
            tangent_dirs[idx] = np.array([1, 0, 0])
    bitangents = np.cross(normals_new, tangent_dirs)
    
    for idx in range(mesh_with_detail.shape[0]):
        mesh_with_detail[idx] += (
            encoded_details[idx, 0] * tangent_dirs[idx] +
            encoded_details[idx, 1] * bitangents[idx] +
            encoded_details[idx, 2] * normals_new[idx]
        )
    return mesh_with_detail

In [ ]:
def make_pos_f(p, v, labels, pos_f_saver, handle_vertex_positions, deformer_fn):

    def pos_f(s,x,y,z, α, β, γ):
        slices = (labels==s)
        r = Rotation.from_euler('xyz', [α, β, γ], degrees=True)
        v_slice = v[slices] + np.array([[x,y,z]])
        center = v_slice.mean(axis=0)
        handle_vertex_positions[slices] = r.apply(v_slice - center) + center
        pos_f_saver[s - 1] = [x,y,z,α,β,γ]
        t0 = time.time()
        v_deformed = pos_f.deformer(handle_vertex_positions)
        p.update_object(vertices = v_deformed)
        t1 = time.time()
        print('FPS', 1/(t1 - t0))
    
    pos_f.deformer = deformer_fn
    return pos_f

In [92]:
pos_f_saver = np.zeros((labels.max() + 1, 6))
def widgets_wrapper():
    segment_widget = iw.Dropdown(options=np.arange(labels.max()) + 1)
    translate_widget = {i:iw.FloatSlider(min=-1, max=1, value=0) 
                        for i in 'xyz'}
    rotate_widget = {a:iw.FloatSlider(min=-90, max=90, value=0, step=1) 
                     for a in 'αβγ'}

    def update_seg(*args):
        (translate_widget['x'].value,translate_widget['y'].value,
        translate_widget['z'].value,
        rotate_widget['α'].value,rotate_widget['β'].value,
        rotate_widget['γ'].value) = pos_f_saver[segment_widget.value]
    segment_widget.observe(update_seg, 'value')
    widgets_dict = dict(s=segment_widget)
    widgets_dict.update(translate_widget)
    widgets_dict.update(rotate_widget)
    return widgets_dict

In [93]:


def original_position_deformer(target_pos):
    return target_pos

def smoothed_position_deformer(target_pos):
    B = remove_hfdetails(target_pos, f, labels)
    return B

B = remove_hfdetails(v, f, labels)
detail_coeffs, edge_ind = compute_detail_vectors(v, B, f)

def smoothed_position_deformer_2(target_pos):
    return deform_smooth_mesh(B, f, labels, target_pos)

def final_position_deformer(target_pos):
    Bp = deform_smooth_mesh(B, f, labels, target_pos)
    Sp = apply_detail_to_deformed(Bp.copy(), f, detail_coeffs, edge_ind)
    return Sp


In [94]:
handle_vertex_positions = v.copy()
pos_f_saver = np.zeros((labels.max() + 1, 6))
color = (labels == 0).astype(int)

p = mp.plot(handle_vertex_positions, f, c=color)
pos_f = make_pos_f(p, v, labels, pos_f_saver, handle_vertex_positions, original_position_deformer)
iw.interact(pos_f, **widgets_wrapper())

p = mp.plot(handle_vertex_positions, f, c=color)
pos_f = make_pos_f(p, v, labels, pos_f_saver, handle_vertex_positions, smoothed_position_deformer)
iw.interact(pos_f, **widgets_wrapper())


p = mp.plot(handle_vertex_positions, f, c=color)
pos_f = make_pos_f(p, v, labels, pos_f_saver, handle_vertex_positions, smoothed_position_deformer_2)
iw.interact(pos_f, **widgets_wrapper())

p = mp.plot(handle_vertex_positions, f, c=color)
pos_f = make_pos_f(p, v, labels, pos_f_saver, handle_vertex_positions, final_position_deformer)
iw.interact(pos_f, **widgets_wrapper())

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.19…

interactive(children=(Dropdown(description='s', options=(1, 2, 3, 4), value=1), FloatSlider(value=0.0, descrip…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.19…

interactive(children=(Dropdown(description='s', options=(1, 2, 3, 4), value=1), FloatSlider(value=0.0, descrip…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.19…

interactive(children=(Dropdown(description='s', options=(1, 2, 3, 4), value=1), FloatSlider(value=0.0, descrip…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.5, 0.19…

interactive(children=(Dropdown(description='s', options=(1, 2, 3, 4), value=1), FloatSlider(value=0.0, descrip…

<function __main__.make_pos_f.<locals>.pos_f(s, x, y, z, α, β, γ)>